# 00 — Silver Layer: Longitudinal Standardization for HCP Classification

This notebook prepares the raw weekly HCP dataset for downstream modeling and analysis.  
The design assumes that the prediction unit is the HCP and that each HCP contributes a fixed 86-week history.

Outputs generated by this notebook:

- `data/silver_layer_longitudinal.parquet`
- `data/hcp_manifest.parquet`
- `data/reports/column_inventory.csv`
- `data/reports/data_quality_summary.csv`
- `data/reports/negative_value_audit.csv`
- `data/reports/hcp_level_target_distribution.csv`

Main principles:

1. preserve weekly structure
2. build HCP-level label metadata without leakage
3. validate temporal integrity
4. create dense categorical views for specialty, state and age band
5. prepare fold assignments at HCP level for future modeling


In [1]:
from pathlib import Path
import warnings
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


## 1. Configuration


In [6]:
DATA_PATH = Path("data/data.csv")
OUTPUT_DIR = Path("data")
REPORT_DIR = OUTPUT_DIR / "reports"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

N_FOLDS = 5
RANDOM_STATE = 42
AUTO_CLIP_CLEARLY_NONNEGATIVE = False  # keep False unless a business rule explicitly confirms clipping


## 2. Data ingestion


In [7]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Input file not found: {DATA_PATH.resolve()}")

try:
    df = pd.read_csv(DATA_PATH, engine="pyarrow")
except Exception:
    df = pd.read_csv(DATA_PATH)

print(f"Raw shape: {df.shape}")
print(df.head(3))


Raw shape: (1800066, 68)
   NUEVO_ID     WEEK_ID  UC_TRX  ORAL_TRX  IL23_TRX  BRAND1_TRX  BRAND2_TRX  \
0     17962  2024-08-02  0.1652    0.0000    0.0000      0.0000      0.0000   
1      3802  2024-11-08  1.5024    0.3780    0.0000      0.0000      0.0000   
2       422  2025-04-25  0.3558    0.1906    0.0000      0.0000      0.0000   

   UC_NRX  ORAL_NRX  IL23_NRX  BRAND1_NRX  BRAND2_NRX  N_CLMBRAND3  \
0  0.0000    0.0000    0.0000      0.0000      0.0000       0.0000   
1  0.0000    0.0000    0.0000      0.0000      0.0000       1.2000   
2  0.5337    0.2859    0.0000      0.0000      0.0000       0.0000   

   N_CLMBRAND1  N_CLMBRAND4  N_CLMBRAND2  N_CLMOTHERS  BRAND1_NBRX  \
0       0.0000       0.0000       0.0000       0.0000       0.0000   
1       0.0000       0.0000       0.0000       2.0000       0.0000   
2       0.0000       0.8000       0.0000       0.0000       0.0000   

   BRAND2_NBRX  ORAL_NBRX  IL23_NBRX  N_CLMBRAND3_NEW  N_CLMBRAND1_NEW  \
0       0.0000     0.0

In [8]:
column_inventory = pd.DataFrame({
    "column": df.columns,
    "dtype_raw": df.dtypes.astype(str).values
})

column_inventory.to_csv(REPORT_DIR / "column_inventory.csv", index=False)
column_inventory


,column,dtype_raw
0,NUEVO_ID,int64
1,WEEK_ID,object
2,UC_TRX,float64
3,ORAL_TRX,float64
4,IL23_TRX,float64
...,...,...
63,"(1960, 1980]",int64
64,"(1980, 2000]",int64
65,"(2000, 2020]",int64
66,"(2020, 2030]",int64


## 3. Canonical column groups

These groups are based on the uploaded dataset.  
The lists are used later for validation, aggregation and reporting.


In [9]:
ID_COL = "NUEVO_ID"
TIME_COL = "WEEK_ID"
TARGET_COL = "ATSEG"

specialty_dummy_cols = ["SPEC_GE", "SPEC_GPFM", "SPEC_IM", "SPEC_NRP", "SPEC_OTHER_SPEC", "SPEC_PHA"]
state_dummy_cols = ["STATE_1", "STATE_2", "STATE_3", "STATE_4", "STATE_5", "STATE_6", "STATE_7", "STATE_8", "STS_OTHER_STS"]
age_dummy_cols = ["(1940, 1960]", "(1960, 1980]", "(1980, 2000]", "(2000, 2020]", "(2020, 2030]"]

calendar_cols = ["YEAR", "QTR", "YEAR_QTR"]

trx_cols = [c for c in df.columns if c.endswith("_TRX")]
nrx_cols = [c for c in df.columns if c.endswith("_NRX")]
nbrx_cols = [c for c in df.columns if c.endswith("_NBRX")]
claim_cols = [c for c in df.columns if c.startswith("N_CLM")]
marketing_cols = ["RTE", "SAMPLES", "COPAY", "DIRECTMAIL", "SPK", "DETAILS"]
rolling_cols = [c for c in df.columns if "_R" in c and c.endswith("SUM")]
gidx_cols = [c for c in df.columns if c.endswith("_GIDX")]

predictor_cols = [
    c for c in df.columns
    if c not in [ID_COL, TIME_COL, TARGET_COL]
]

dynamic_numeric_cols = [
    c for c in (trx_cols + nrx_cols + nbrx_cols + claim_cols + marketing_cols + rolling_cols + gidx_cols + calendar_cols)
    if c in df.columns
]

static_dummy_cols = [c for c in (specialty_dummy_cols + state_dummy_cols + age_dummy_cols) if c in df.columns]

print("Dynamic numeric columns:", len(dynamic_numeric_cols))
print("Static dummy columns:", len(static_dummy_cols))


Dynamic numeric columns: 45
Static dummy columns: 20


## 4. Type standardization and HCP-level label propagation


In [10]:
df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
if df[TIME_COL].isna().any():
    raise ValueError("WEEK_ID contains invalid dates after parsing.")

df = df.sort_values([ID_COL, TIME_COL]).reset_index(drop=True)

# Preserve raw target and create HCP-level propagated target
df["ATSEG_RAW"] = df[TARGET_COL]

hcp_target_map = (
    df[[ID_COL, TARGET_COL]]
    .dropna(subset=[TARGET_COL])
    .drop_duplicates(subset=[ID_COL])
    .set_index(ID_COL)[TARGET_COL]
    .to_dict()
)

df["ATSEG_HCP"] = df[ID_COL].map(hcp_target_map)
df["IS_LABELED_HCP"] = df["ATSEG_HCP"].notna().astype("int8")
df["WEEK_IDX"] = df.groupby(ID_COL).cumcount().astype("int16")

# Downcast numerics where appropriate
float64_cols = df.select_dtypes(include=["float64"]).columns.tolist()
int64_cols = df.select_dtypes(include=["int64"]).columns.tolist()

for col in float64_cols:
    df[col] = pd.to_numeric(df[col], downcast="float")

for col in int64_cols:
    if col in [ID_COL, "YEAR_QTR"]:
        continue
    df[col] = pd.to_numeric(df[col], downcast="integer")

print(df[[ID_COL, TIME_COL, TARGET_COL, "ATSEG_HCP", "IS_LABELED_HCP", "WEEK_IDX"]].head())


   NUEVO_ID    WEEK_ID ATSEG ATSEG_HCP  IS_LABELED_HCP  WEEK_IDX
0         1 2024-01-05  None       NaN               0         0
1         1 2024-01-12  None       NaN               0         1
2         1 2024-01-19  None       NaN               0         2
3         1 2024-01-26  None       NaN               0         3
4         1 2024-02-02  None       NaN               0         4


## 5. Dense categorical views from one-hot groups

The original dummy columns are preserved.  
Dense views are added for readability and downstream EDA.


In [11]:
def recover_category_from_dummies(dataframe, columns, output_col, mapping=None):
    cols = [c for c in columns if c in dataframe.columns]
    if not cols:
        dataframe[output_col] = pd.Series([np.nan] * len(dataframe), index=dataframe.index, dtype="object")
        return dataframe

    tmp = dataframe[cols].copy()
    row_sum = tmp.sum(axis=1)
    winner = tmp.idxmax(axis=1)

    if mapping is not None:
        winner = winner.map(mapping)

    dataframe[output_col] = np.where(row_sum > 0, winner, np.nan)
    dataframe[output_col] = pd.Series(dataframe[output_col], index=dataframe.index, dtype="object")
    return dataframe

specialty_mapping = {
    "SPEC_GE": "GE",
    "SPEC_GPFM": "GPFM",
    "SPEC_IM": "IM",
    "SPEC_NRP": "NRP",
    "SPEC_OTHER_SPEC": "OTHER_SPEC",
    "SPEC_PHA": "PHA",
}

state_mapping = {
    "STATE_1": "STATE_1",
    "STATE_2": "STATE_2",
    "STATE_3": "STATE_3",
    "STATE_4": "STATE_4",
    "STATE_5": "STATE_5",
    "STATE_6": "STATE_6",
    "STATE_7": "STATE_7",
    "STATE_8": "STATE_8",
    "STS_OTHER_STS": "OTHER_STATE",
}

age_mapping = {
    "(1940, 1960]": "(1940, 1960]",
    "(1960, 1980]": "(1960, 1980]",
    "(1980, 2000]": "(1980, 2000]",
    "(2000, 2020]": "(2000, 2020]",
    "(2020, 2030]": "(2020, 2030]",
}

df = recover_category_from_dummies(df, specialty_dummy_cols, "SPECIALTY_GROUP", specialty_mapping)
df = recover_category_from_dummies(df, state_dummy_cols, "STATE_GROUP", state_mapping)
df = recover_category_from_dummies(df, age_dummy_cols, "AGE_RANGE_GROUP", age_mapping)

df[["SPECIALTY_GROUP", "STATE_GROUP", "AGE_RANGE_GROUP"]].head()


,SPECIALTY_GROUP,STATE_GROUP,AGE_RANGE_GROUP
0,PHA,OTHER_STATE,"(1940, 1960]"
1,PHA,OTHER_STATE,"(1940, 1960]"
2,PHA,OTHER_STATE,"(1940, 1960]"
3,PHA,OTHER_STATE,"(1940, 1960]"
4,PHA,OTHER_STATE,"(1940, 1960]"


## 6. Data quality audit


In [12]:
data_quality_summary = []

data_quality_summary.append({
    "check": "raw_rows",
    "value": int(df.shape[0])
})
data_quality_summary.append({
    "check": "raw_columns",
    "value": int(df.shape[1])
})
data_quality_summary.append({
    "check": "unique_hcps",
    "value": int(df[ID_COL].nunique())
})
data_quality_summary.append({
    "check": "unique_weeks",
    "value": int(df[TIME_COL].nunique())
})
data_quality_summary.append({
    "check": "feature_missing_rate_max_excluding_target",
    "value": float(df.drop(columns=[TARGET_COL, "ATSEG_RAW", "ATSEG_HCP"], errors="ignore").isna().mean().max())
})
data_quality_summary.append({
    "check": "target_missing_rate_row_level",
    "value": float(df[TARGET_COL].isna().mean())
})

duplicates = df.duplicated([ID_COL, TIME_COL]).sum()
data_quality_summary.append({
    "check": "duplicate_hcp_week_pairs",
    "value": int(duplicates)
})

rows_per_hcp = df.groupby(ID_COL).size()
weeks_per_hcp = df.groupby(ID_COL)[TIME_COL].nunique()

data_quality_summary.append({
    "check": "rows_per_hcp_min",
    "value": int(rows_per_hcp.min())
})
data_quality_summary.append({
    "check": "rows_per_hcp_median",
    "value": float(rows_per_hcp.median())
})
data_quality_summary.append({
    "check": "rows_per_hcp_max",
    "value": int(rows_per_hcp.max())
})
data_quality_summary.append({
    "check": "weeks_per_hcp_min",
    "value": int(weeks_per_hcp.min())
})
data_quality_summary.append({
    "check": "weeks_per_hcp_median",
    "value": float(weeks_per_hcp.median())
})
data_quality_summary.append({
    "check": "weeks_per_hcp_max",
    "value": int(weeks_per_hcp.max())
})

data_quality_summary = pd.DataFrame(data_quality_summary)
data_quality_summary.to_csv(REPORT_DIR / "data_quality_summary.csv", index=False)
data_quality_summary


,check,value
0,raw_rows,"1,800,066.0000"
1,raw_columns,75.0000
2,unique_hcps,"20,931.0000"
3,unique_weeks,86.0000
4,feature_missing_rate_max_excluding_target,0.1780
5,target_missing_rate_row_level,0.4315
6,duplicate_hcp_week_pairs,0.0000
7,rows_per_hcp_min,86.0000
8,rows_per_hcp_median,86.0000
9,rows_per_hcp_max,86.0000


In [13]:
# Label consistency at HCP level
hcp_label_consistency = (
    df.groupby(ID_COL)["ATSEG_HCP"]
    .nunique(dropna=True)
    .value_counts(dropna=False)
    .rename_axis("n_unique_labels")
    .reset_index(name="hcp_count")
)

print(hcp_label_consistency)

# Weekly continuity audit
date_gap_audit = (
    df.groupby(ID_COL)[TIME_COL]
    .diff()
    .dropna()
    .dt.days
    .value_counts()
    .sort_index()
    .rename_axis("day_gap")
    .reset_index(name="count")
)

date_gap_audit.head(10)


   n_unique_labels  hcp_count
0                1      11899
1                0       9032


,day_gap,count
0,7,1779135


## 7. Negative value audit

The dataset contains some index variables (`*_GIDX`) that can legitimately take negative values, so no blanket clipping is performed.

Only clearly non-negative operational variables are audited below.  
Examples: TRX, NRX, NBRX, claim counts, rolling sums, and promotional contact volumes.


In [14]:
clearly_nonnegative_cols = [
    c for c in (trx_cols + nrx_cols + nbrx_cols + claim_cols + marketing_cols + rolling_cols)
    if c in df.columns
]

negative_value_audit = pd.DataFrame({
    "column": clearly_nonnegative_cols,
    "negative_count": [int((df[c] < 0).sum()) for c in clearly_nonnegative_cols],
    "minimum_value": [float(df[c].min()) for c in clearly_nonnegative_cols]
}).sort_values(["negative_count", "minimum_value"], ascending=[False, True])

negative_value_audit.to_csv(REPORT_DIR / "negative_value_audit.csv", index=False)
negative_value_audit.head(20)


,column,negative_count,minimum_value
6,ORAL_NRX,1,-0.0003
0,UC_TRX,0,0.0000
1,ORAL_TRX,0,0.0000
2,IL23_TRX,0,0.0000
3,BRAND1_TRX,0,0.0000
4,BRAND2_TRX,0,0.0000
5,UC_NRX,0,0.0000
7,IL23_NRX,0,0.0000
8,BRAND1_NRX,0,0.0000
9,BRAND2_NRX,0,0.0000


In [15]:
if AUTO_CLIP_CLEARLY_NONNEGATIVE:
    cols_to_clip = negative_value_audit.loc[negative_value_audit["negative_count"] > 0, "column"].tolist()
    for col in cols_to_clip:
        df[col] = df[col].clip(lower=0)
    print(f"Clipped {len(cols_to_clip)} clearly non-negative columns.")
else:
    print("AUTO_CLIP_CLEARLY_NONNEGATIVE = False. No clipping was applied.")


AUTO_CLIP_CLEARLY_NONNEGATIVE = False. No clipping was applied.


## 8. HCP manifest and fold assignment

Fold assignment is created at HCP level to preserve the correct prediction unit for future modeling.


In [16]:
hcp_manifest = (
    df.groupby(ID_COL)
    .agg(
        n_rows=(ID_COL, "size"),
        n_weeks=(TIME_COL, "nunique"),
        first_week=(TIME_COL, "min"),
        last_week=(TIME_COL, "max"),
        ATSEG_HCP=("ATSEG_HCP", "first"),
        IS_LABELED_HCP=("IS_LABELED_HCP", "max"),
        SPECIALTY_GROUP=("SPECIALTY_GROUP", "first"),
        STATE_GROUP=("STATE_GROUP", "first"),
        AGE_RANGE_GROUP=("AGE_RANGE_GROUP", "first"),
    )
    .reset_index()
)

hcp_manifest["HCP_FOLD"] = -1

labeled_mask = hcp_manifest["IS_LABELED_HCP"] == 1
labeled_manifest = hcp_manifest.loc[labeled_mask].copy()

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
for fold, (_, valid_idx) in enumerate(skf.split(labeled_manifest[[ID_COL]], labeled_manifest["ATSEG_HCP"])):
    labeled_manifest.iloc[valid_idx, labeled_manifest.columns.get_loc("HCP_FOLD")] = fold

hcp_manifest.loc[labeled_mask, "HCP_FOLD"] = labeled_manifest["HCP_FOLD"].values

fold_map = dict(zip(hcp_manifest[ID_COL], hcp_manifest["HCP_FOLD"]))
df["HCP_FOLD"] = df[ID_COL].map(fold_map).fillna(-1).astype("int8")

hcp_target_distribution = (
    hcp_manifest["ATSEG_HCP"]
    .value_counts(dropna=False)
    .rename_axis("ATSEG_HCP")
    .reset_index(name="hcp_count")
)

hcp_target_distribution.to_csv(REPORT_DIR / "hcp_level_target_distribution.csv", index=False)

print(hcp_manifest.head())
print(hcp_target_distribution)


   NUEVO_ID  n_rows  n_weeks first_week  last_week ATSEG_HCP  IS_LABELED_HCP  \
0         1      86       86 2024-01-05 2025-08-22      None               0   
1         2      86       86 2024-01-05 2025-08-22      None               0   
2         3      86       86 2024-01-05 2025-08-22     SEG_A               1   
3         4      86       86 2024-01-05 2025-08-22      None               0   
4         5      86       86 2024-01-05 2025-08-22      None               0   

  SPECIALTY_GROUP  STATE_GROUP AGE_RANGE_GROUP  HCP_FOLD  
0             PHA  OTHER_STATE    (1940, 1960]        -1  
1              IM  OTHER_STATE            None        -1  
2              GE  OTHER_STATE    (1940, 1960]         1  
3              IM      STATE_7    (1940, 1960]        -1  
4              GE  OTHER_STATE    (1960, 1980]        -1  
  ATSEG_HCP  hcp_count
0      None       9032
1     SEG_A       6406
2     SEG_B       3349
3     SEG_C       2144


## 9. Persist curated assets


In [17]:
silver_path = OUTPUT_DIR / "silver_layer_longitudinal.parquet"
manifest_path = OUTPUT_DIR / "hcp_manifest.parquet"

df.to_parquet(silver_path, index=False)
hcp_manifest.to_parquet(manifest_path, index=False)

print(f"Saved silver dataset to: {silver_path.resolve()}")
print(f"Saved HCP manifest to: {manifest_path.resolve()}")


Saved silver dataset to: /Users/davidbazalduamendez/Documents/GitHub/Pfizer-segmentation-Ulcerative-Colitis/models/Lightbm/data/silver_layer_longitudinal.parquet
Saved HCP manifest to: /Users/davidbazalduamendez/Documents/GitHub/Pfizer-segmentation-Ulcerative-Colitis/models/Lightbm/data/hcp_manifest.parquet


## 10. Final review


In [18]:
print("Silver dataset shape:", df.shape)
print("HCP manifest shape:", hcp_manifest.shape)
print(df[[ID_COL, TIME_COL, "WEEK_IDX", "ATSEG_HCP", "IS_LABELED_HCP", "HCP_FOLD", "SPECIALTY_GROUP", "STATE_GROUP", "AGE_RANGE_GROUP"]].head(10))


Silver dataset shape: (1800066, 76)
HCP manifest shape: (20931, 11)
   NUEVO_ID    WEEK_ID  WEEK_IDX ATSEG_HCP  IS_LABELED_HCP  HCP_FOLD  \
0         1 2024-01-05         0       NaN               0        -1   
1         1 2024-01-12         1       NaN               0        -1   
2         1 2024-01-19         2       NaN               0        -1   
3         1 2024-01-26         3       NaN               0        -1   
4         1 2024-02-02         4       NaN               0        -1   
5         1 2024-02-09         5       NaN               0        -1   
6         1 2024-02-16         6       NaN               0        -1   
7         1 2024-02-23         7       NaN               0        -1   
8         1 2024-03-01         8       NaN               0        -1   
9         1 2024-03-08         9       NaN               0        -1   

  SPECIALTY_GROUP  STATE_GROUP AGE_RANGE_GROUP  
0             PHA  OTHER_STATE    (1940, 1960]  
1             PHA  OTHER_STATE    (1940, 